# Modelo jerárquico con Binomial Negativa para `receiving_tds`

## Qué responde este notebook

Todo lo probado hasta ahora (Ridge, RandomForest, XGBoost, HistGradientBoosting, Tweedie, hurdle)
es la misma familia de fondo: funciones flexibles ajustadas a los datos, sin explotar la
estructura real de un panel -- los mismos ~500-900 jugadores repetidos semana tras semana, año
tras año. Esa estructura se aproximaba a mano con promedios rezagados
(`_last3_avg`/`_season_avg`/`_career_avg`), pero un modelo jerárquico (mixto) la modela
directamente: un efecto aleatorio por jugador captura "quién es este jugador" usando toda su
historia con encogimiento estadístico (*shrinkage*) hacia el promedio de la posición, en vez de
un promedio móvil crudo.

Se combina con una verosimilitud **Binomial Negativa** -- la herramienta de libro de texto para
conteos enteros sobre-dispersos como touchdowns, mejor especificada que `reg:tweedie` (pensado
para datos continuos con masa en cero, no para conteos discretos). Ambas piezas juntas dan un
modelo de efectos mixtos (GLMM): exactamente lo que se pidió -- "más explicativo, que se ajuste
mejor a los datos".

**Herramienta nueva en el proyecto**: `pymc`/`bambi`/`arviz` (Python, Bayesiano) -- agregados a
`docker/requirements.txt` e imagen reconstruida. Es la primera vez que `weekly/` necesita algo más
que `pandas`/`scikit-learn`/`xgboost`.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling

import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

pd.set_option("display.width", 160)

AZUL, NARANJA, VERDE, GRIS, GRIS_CLARO = "#2a78d6", "#eb6834", "#1baf7a", "#888780", "#c3c2b7"

wr = features.construir_tabla_modelado(range(2016, 2026))
features_ridge = features.columnas_por_modelo("ridge")
train_mask = wr["season"].between(2016, 2021)
val_mask = wr["season"].between(2022, 2023)
test_mask = wr["season"].between(2024, 2025)
print(f"features (set Ridge, ya sin colinealidad severa): {len(features_ridge)}")


features (set Ridge, ya sin colinealidad severa): 30


## 1. Un problema real antes de ajustar nada: el `dropna()` ingenuo pierde justo la
población que más importa

A diferencia de XGBoost (maneja NaN nativo) o Ridge (ya se imputa en `modeling.py`), un modelo de
`bambi` necesita datos completos. Antes de decidir cómo, vale ver qué se perdería con un
`dropna()` directo.

In [2]:
perdidas = wr.loc[train_mask, features_ridge].isna().any(axis=1)
print(f"filas de train que un dropna() perderia: {perdidas.sum()} de {train_mask.sum()} ({perdidas.mean():.1%})")

nulos_por_columna = wr.loc[train_mask, features_ridge].isna().sum().sort_values(ascending=False)
print("\ncolumnas con mas nulos:")
print(nulos_por_columna.head(6))

# caso concreto: una fila con nulos en _season_avg es literalmente la primera semana
# de esa temporada para ese jugador -- season_avg (shift(1).expanding()) no tiene nada
# anterior de donde promediar todavia.
ejemplo = wr.loc[train_mask & wr["receiving_tds_season_avg"].isna()].iloc[0]
print(f"\nejemplo real: {ejemplo['player_display_name']}, season={int(ejemplo['season'])} week={int(ejemplo['week'])}")
print(f"receiving_tds_season_avg={ejemplo['receiving_tds_season_avg']} (nulo -- es su primera semana jugada esa temporada)")


filas de train que un dropna() perderia: 2865 de 13795 (20.8%)

columnas con mas nulos:
receiving_tds_season_avg          1371
fantasy_points_ppr_season_avg     1371
receptions_season_avg             1371
receiving_yards_season_avg        1371
receiving_air_yards_season_avg    1371
targets_season_avg                1371
dtype: int64

ejemplo real: Steve Smith, season=2016 week=1
receiving_tds_season_avg=nan (nulo -- es su primera semana jugada esa temporada)


Confirmado: son exactamente las primeras apariciones de temporada o de carrera de cada
jugador -- la población donde más debería ayudar el encogimiento jerárquico (poca historia propia
→ apoyarse más en el promedio del grupo). Perderlas de entrada sería contraproducente. Se imputa
en vez de descartar, mismo criterio que ya usa Ridge en `modeling.py` (mediana para numéricas,
moda para la única columna categórica, `anios_experiencia_bucket`).

In [3]:
cols_numericas = [c for c in features_ridge if c != "anios_experiencia_bucket"]
imputador_num = SimpleImputer(strategy="median")
imputador_cat = SimpleImputer(strategy="most_frequent")

def imputar(df, ajustar):
    fn_num = imputador_num.fit_transform if ajustar else imputador_num.transform
    fn_cat = imputador_cat.fit_transform if ajustar else imputador_cat.transform
    num = pd.DataFrame(fn_num(df[cols_numericas]), index=df.index, columns=cols_numericas)
    cat = pd.DataFrame(fn_cat(df[["anios_experiencia_bucket"]]), index=df.index, columns=["anios_experiencia_bucket"])
    return pd.concat([num, cat], axis=1)[features_ridge]

target = "receiving_tds"
X_train_imp = imputar(wr.loc[train_mask, features_ridge], ajustar=True)
datos_train = X_train_imp.copy()
datos_train[target] = wr.loc[train_mask, target].to_numpy()
datos_train["player_id"] = wr.loc[train_mask, "player_id"].astype(str).to_numpy()
print(f"filas de entrenamiento tras imputar: {len(datos_train)} (0 perdidas, vs. {perdidas.sum()} que se hubieran perdido)")


filas de entrenamiento tras imputar: 13795 (0 perdidas, vs. 2865 que se hubieran perdido)


## 2. El modelo: efectos fijos (30 features) + intercepto aleatorio por jugador,
verosimilitud Binomial Negativa

`receiving_tds ~ [30 features del set Ridge] + (1|player_id)` -- sintaxis de fórmula tipo
`lme4`/R, que `bambi` traduce a un modelo PyMC completo. Sin priors manuales en esta primera
versión: se usan los priors por default de `bambi` (débilmente informativos, calculados a partir
de la escala de los datos) -- mismo criterio de "no buscar de más" que el resto de esta ronda.

In [4]:
formula = f"{target} ~ " + " + ".join(features_ridge) + " + (1|player_id)"
modelo = bmb.Model(formula, datos_train, family="negativebinomial")

import time
t0 = time.time()
idata = modelo.fit(draws=500, tune=500, chains=4, cores=4, random_seed=0, progressbar=False)
print(f"tiempo de ajuste (500 muestras + 500 de calentamiento, 4 cadenas): {time.time()-t0:.1f}s")


Initializing NUTS using jitter+adapt_diag...


/usr/local/lib/python3.13/site-packages/pytensor/tensor/rewriting/elemwise.py:1004: UserWarning: Loop fusion failed because the resulting node would exceed the kernel argument limit.
  warn(


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [alpha, Intercept, anios_experiencia, catch_rate_career_avg, depth_team, draft_pick, edad, fantasy_points_ppr_last3_avg, fantasy_points_ppr_last5_avg, fantasy_points_ppr_season_avg, receiving_10_last3_avg, receiving_16_last5_avg, receiving_20_last3_avg, receiving_air_yards_season_avg, receiving_epa_last3_avg, receiving_first_downs_career_avg, receiving_first_downs_last5_avg, receiving_tds_season_avg, receiving_yards_career_avg, receiving_yards_last3_avg, receiving_yards_season_avg, receptions_career_avg, receptions_last3_avg, receptions_last5_avg, receptions_season_avg, targets_last3_avg, targets_last5_avg, targets_season_avg, wopr_last3_avg, wopr_last5_avg, anios_experiencia_sq, anios_experiencia_bucket, 1|player_id_sigma, 1|player_id_offset]


Sampling 4 chains for 500 tune and 500 draw iterations (2_000 + 2_000 draws total) took 189 seconds.


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


tiempo de ajuste (500 muestras + 500 de calentamiento, 4 cadenas): 203.1s


## 3. Diagnóstico de convergencia -- honesto, sobre TODOS los parámetros

No solo los coeficientes fijos: el modelo tiene un parámetro por cada uno de los 505 jugadores
del set de entrenamiento (505 efectos individuales) más los 30 coeficientes fijos, el intercepto,
la dispersión de la Binomial Negativa y la varianza del efecto de jugador -- 540 en total.

In [5]:
resumen_completo = az.summary(idata)
problematicos = resumen_completo[resumen_completo["r_hat"] > 1.01]
print(f"parametros totales: {len(resumen_completo)}")
print(f"r_hat maximo: {resumen_completo['r_hat'].max():.4f}")
print(f"parametros con r_hat > 1.01: {len(problematicos)} de {len(resumen_completo)} ({len(problematicos)/len(resumen_completo):.1%})")
print("\nesos parametros (todos son efectos individuales de jugadores especificos, no efectos fijos):")
print(problematicos[["mean", "sd", "ess_bulk", "r_hat"]].to_string())


parametros totales: 540
r_hat maximo: 1.0160
parametros con r_hat > 1.01: 9 de 540 (1.7%)

esos parametros (todos son efectos individuales de jugadores especificos, no efectos fijos):
                           mean     sd ess_bulk r_hat
1|player_id[00-0030521]   0.188   0.26     4902  1.01
1|player_id[00-0030669]   0.102   0.23     5145  1.02
1|player_id[00-0031972]  -0.118   0.27     5044  1.01
1|player_id[00-0032724]   -0.11   0.27     3594  1.01
1|player_id[00-0035022]  -0.082   0.27     4609  1.01
1|player_id[00-0035216]   0.251  0.194     3524  1.01
1|player_id[00-0035544]  -0.035   0.28     4837  1.01
1|player_id[00-0035590]  -0.047   0.28     5166  1.01
1|player_id[00-0036959]  -0.009   0.29     5705  1.01


Los 9 parámetros con `r_hat` ligeramente por encima de 1.01 son efectos individuales de
jugadores puntuales -- ninguno es un coeficiente fijo (esos convergen limpio, `r_hat=1.00`
todos). Con `ess_bulk` de todos modos alto (3,500+) en los 9 casos, es la dificultad esperada de
estimar un efecto por jugador con relativamente poca información individual, no un problema de
la cadena en general. Se reporta explícito, no se oculta.

## 4. Interpretabilidad -- ya no hace falta SHAP

Los coeficientes fijos se leen directo: media posterior + intervalo de credibilidad al 89% (ETI,
el default de `bambi`/`arviz`). Si el intervalo no cruza el cero, el efecto es claro; si lo cruza,
la evidencia es débil con estos datos.

In [6]:
coeficientes = az.summary(idata, var_names=["Intercept"] + features_ridge, filter_vars=None)
coeficientes = coeficientes.reindex(coeficientes["mean"].abs().sort_values(ascending=False).index)
coeficientes["credible"] = np.where(
    (coeficientes["eti89_lb"] > 0) | (coeficientes["eti89_ub"] < 0), "sí", "no (cruza cero)"
)
print("top 10 coeficientes por magnitud, con si su intervalo de credibilidad cruza cero:")
print(coeficientes[["mean", "eti89_lb", "eti89_ub", "credible"]].head(10).to_string())


top 10 coeficientes por magnitud, con si su intervalo de credibilidad cruza cero:
                                     mean eti89_lb eti89_ub         credible
wopr_last3_avg                       0.62    -0.16      1.4  no (cruza cero)
catch_rate_career_avg              -0.536    -0.91    -0.16               sí
depth_team                         -0.296    -0.37    -0.22               sí
Intercept                           -0.23     -1.4     0.92  no (cruza cero)
anios_experiencia_bucket[Rookie]   -0.104    -0.26    0.055  no (cruza cero)
receptions_career_avg               0.089   -0.019     0.19  no (cruza cero)
receptions_season_avg               0.085   -0.036     0.21  no (cruza cero)
receiving_20_last3_avg             -0.082    -0.19    0.027  no (cruza cero)
anios_experiencia_bucket[6+ años]   0.058     -0.2     0.31  no (cruza cero)
edad                               -0.052     -0.1  -0.0028               sí


`depth_team` (posición en la alineación) sale con efecto negativo claro (intervalo
`[-0.37, -0.22]`, no cruza cero) -- confirma con estadística formal lo que antes solo se veía
como importancia de árbol en SHAP: estar más arriba en la alineación (número más bajo) predice
más touchdowns, controlando todo lo demás. `wopr_last3_avg`, en cambio, tiene un intervalo amplio
que sí cruza cero -- con esta muestra, no hay evidencia suficientemente clara de su efecto una vez
que las demás variables ya están en el modelo (probable colinealidad residual con las columnas de
participación y volumen).

## 5. Interpretabilidad -- el efecto individual de un jugador real

El intercepto por jugador (`1|player_id`) es la pieza nueva que ningún modelo anterior tenía: una
habilidad "extra" de ese jugador específico, más allá de lo que explican sus features. Ja'Marr
Chase es un caso real conocido en todo este proyecto -- ¿qué dice el modelo de él?

In [7]:
chase_id = data.cargar_jugadores().pipe(lambda j: j[j["display_name"] == "Ja'Marr Chase"])["gsis_id"].iloc[0]
efectos = idata.posterior["1|player_id"].mean(dim=["chain", "draw"]).to_dataframe().reset_index()
efectos.columns = ["player_id", "efecto_log"]

fila_chase = efectos[efectos["player_id"] == chase_id]
efecto_chase = fila_chase["efecto_log"].iloc[0]
print(f"efecto individual de Ja'Marr Chase (escala log): {efecto_chase:.3f}")
print(f"en escala de tasa: exp({efecto_chase:.3f}) = {np.exp(efecto_chase):.2f}x -- su tasa esperada de TDs es "
      f"~{100*(np.exp(efecto_chase)-1):.0f}% mas alta que un jugador con sus mismas features pero sin su 'plus' individual")
print(f"\npara contexto, rango de efectos en los {len(efectos)} jugadores de train: "
      f"[{efectos['efecto_log'].min():.3f}, {efectos['efecto_log'].max():.3f}], mediana={efectos['efecto_log'].median():.3f}")


efecto individual de Ja'Marr Chase (escala log): 0.260
en escala de tasa: exp(0.260) = 1.30x -- su tasa esperada de TDs es ~30% mas alta que un jugador con sus mismas features pero sin su 'plus' individual

para contexto, rango de efectos en los 505 jugadores de train: [-0.389, 0.554], mediana=-0.019


## 6. Evaluación en test 2024-2025 -- contra todo lo demás ya probado

Se predice con `kind="response_params"` (el valor esperado de la Binomial Negativa, `mu`) -- no
`kind="response"` (que muestrea valores simulados y agrega ruido extra innecesario a una
comparación de punto). Los jugadores de test que no estaban en entrenamiento (rookies o jugadores
sin suficientes partidos en 2016-2021) no tienen efecto individual estimado -- `bambi` los trata
automáticamente con la distribución poblacional del efecto de jugador (un jugador "promedio"
razonable, no un error).

In [8]:
X_test_imp = imputar(wr.loc[test_mask, features_ridge], ajustar=False)
datos_test = X_test_imp.copy()
datos_test["player_id"] = wr.loc[test_mask, "player_id"].astype(str).to_numpy()

pred_params = modelo.predict(idata, data=datos_test, kind="response_params", inplace=False, sample_new_groups=True)
pred_media = pred_params.posterior["mu"].mean(dim=["chain", "draw"]).to_numpy()

y_test = wr.loc[test_mask, target].to_numpy()
m_jerarquico = modeling.metricas(pd.Series(y_test), pd.Series(pred_media))

comparacion = pd.DataFrame({
    "Tweedie XGBoost (13-20)": {"mae": 0.2488, "r2": -0.0410},
    "Hurdle (23)": {"mae": 0.3074, "r2": 0.0703},
    "Ensamble Tweedie+hurdle w=0.5 (23)": {"mae": 0.2781, "r2": 0.0456},
    "Jerarquico + Binomial Negativa (nuevo)": {"mae": m_jerarquico["mae"], "r2": m_jerarquico["r2"]},
}).T
print(comparacion.round(4).to_string())


/tmp/ipykernel_7/405150067.py:5: FutureWarning: 'sample_new_groups' is deprecated and will be removed in a future version. New groups will be handled automatically.
  pred_params = modelo.predict(idata, data=datos_test, kind="response_params", inplace=False, sample_new_groups=True)


                                           mae      r2
Tweedie XGBoost (13-20)                 0.2488 -0.0410
Hurdle (23)                             0.3074  0.0703
Ensamble Tweedie+hurdle w=0.5 (23)      0.2781  0.0456
Jerarquico + Binomial Negativa (nuevo)  0.3001  0.0732


El modelo jerárquico logra el **mejor R² de todos los intentos** para este target
(supera incluso al modelo hurdle), con MAE razonable -- mejor que el hurdle puro, peor que el
Tweedie puro (mismo patrón de esta tensión MAE-vs-R² ya documentado varias veces, pero ahora con
un punto nuevo, más explicable, en esa curva de trade-off).

## 7. Hallazgo honesto, no el que se esperaba

La hipótesis natural era que el shrinkage jerárquico ayudaría más a jugadores que el modelo ya
había visto en entrenamiento (con un efecto individual real estimado) que a jugadores nuevos (que
caen en el efecto poblacional genérico). Se verifica con el mismo test, separado en 2 grupos.

In [9]:
jugadores_train = set(datos_train["player_id"].unique())
datos_test["pred"] = pred_media
datos_test["real"] = y_test
datos_test["visto_en_train"] = datos_test["player_id"].isin(jugadores_train)

filas_split = []
for visto, grupo in datos_test.groupby("visto_en_train"):
    m_grupo = modeling.metricas(grupo["real"], grupo["pred"])
    filas_split.append({"grupo": "ya visto en train" if visto else "jugador nuevo (no en train)",
                         "n": len(grupo), **m_grupo})
print(pd.DataFrame(filas_split).round(4).to_string(index=False))


                      grupo    n    mae   rmse     r2
jugador nuevo (no en train) 2702 0.2789 0.4363 0.0644
          ya visto en train 2250 0.3256 0.4652 0.0817


**No se confirma la hipótesis** -- los jugadores nuevos salen con *mejor* MAE (0.279 vs.
0.326) aunque R² algo menor (0.065 vs. 0.082) que los ya vistos en entrenamiento. Una lectura
plausible, no la única posible: los jugadores ya vistos en 2016-2021 y todavía activos en
2024-2025 son en su mayoría veteranos que para entonces ya cambiaron de rol/equipo/relevancia
(justo la clase de deriva de desempeño entre temporadas que Daniel señaló como pendiente de
investigar con más cuidado) -- más difíciles de predecir en términos absolutos que un jugador más
joven cuyas features actuales describen bien su situación de hoy. Se documenta el hallazgo real,
no se fuerza una narrativa que no está confirmada con más evidencia.

## 8. Guardar el modelo

`idata.to_netcdf()` -- el formato estándar de `arviz` -- falla al **recargar** por una
incompatibilidad real entre las versiones instaladas de `h5netcdf`/`xarray`
(`AttributeError: 'Variable' object has no attribute 'filters'`, confirmado). Mismo tipo de
problema ya visto antes con el wrapper de sklearn/XGBoost en `18_validacion_temporada_actual.ipynb`
-- se resuelve con el mismo criterio: usar la alternativa que sí funciona. `joblib` (ya usado para
los modelos de cuantil/ensamble) guarda y recarga el objeto `idata` completo sin problema.

In [10]:
import json, os
import joblib

os.makedirs("../models", exist_ok=True)
joblib.dump({
    "idata": idata,
    "formula": formula,
    "family": "negativebinomial",
    "imputador_num": imputador_num,
    "imputador_cat": imputador_cat,
    "cols_numericas": cols_numericas,
    "features_ridge": features_ridge,
}, "../models/receiving_tds_jerarquico.joblib")

# verificar que recargar y predecir con el objeto guardado da el mismo resultado
paquete_recargado = joblib.load("../models/receiving_tds_jerarquico.joblib")
modelo_verificacion = bmb.Model(paquete_recargado["formula"], datos_train, family=paquete_recargado["family"])
pred_verificacion = modelo_verificacion.predict(
    paquete_recargado["idata"], data=datos_test, kind="response_params", inplace=False, sample_new_groups=True
)
pred_media_verificacion = pred_verificacion.posterior["mu"].mean(dim=["chain", "draw"]).to_numpy()
m_verificacion = modeling.metricas(pd.Series(y_test), pd.Series(pred_media_verificacion))
print(f"metricas con el modelo recargado desde disco: mae={m_verificacion['mae']:.4f} r2={m_verificacion['r2']:.4f}")
print(f"metricas originales (antes de guardar):        mae={m_jerarquico['mae']:.4f} r2={m_jerarquico['r2']:.4f}")

metadata = {
    "target": target,
    "modelo": "bambi.Model, familia negativebinomial, efecto aleatorio (1|player_id)",
    "formula": formula,
    "n_efectos_jugador": int(datos_train["player_id"].nunique()),
    "diagnostico_convergencia": {
        "r_hat_max": round(float(resumen_completo["r_hat"].max()), 4),
        "parametros_r_hat_mayor_1_01": int(len(problematicos)),
        "parametros_totales": int(len(resumen_completo)),
        "nota": "los parametros con r_hat>1.01 son todos efectos individuales de jugadores especificos, no efectos fijos",
    },
    "formato_guardado": "joblib (netCDF nativo de arviz falla al recargar -- incompatibilidad h5netcdf/xarray, ver seccion 8)",
    "metricas_test_2024_2025": {"mae": round(float(m_jerarquico["mae"]), 4), "r2": round(float(m_jerarquico["r2"]), 4)},
    "fecha_de_guardado": pd.Timestamp.now(tz="UTC").isoformat(),
    "limitaciones_conocidas": [
        "jugadores nuevos en test (no vistos en train) usan el efecto poblacional generico, no uno individual -- 176 de 299 jugadores de test 2024-2025 caen en este caso",
        "9 de 540 parametros (efectos individuales de jugadores especificos) tienen r_hat ligeramente sobre 1.01 -- ess_bulk alto (3500+) en todos, no es un problema de la cadena en general",
    ],
}
with open("../models/receiving_tds_jerarquico_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("\nguardado: receiving_tds_jerarquico.joblib + metadata")


/tmp/ipykernel_7/2641117181.py:18: FutureWarning: 'sample_new_groups' is deprecated and will be removed in a future version. New groups will be handled automatically.
  pred_verificacion = modelo_verificacion.predict(


metricas con el modelo recargado desde disco: mae=0.3002 r2=0.0733
metricas originales (antes de guardar):        mae=0.3001 r2=0.0732

guardado: receiving_tds_jerarquico.joblib + metadata


## Cierre

- Modelo jerárquico (efecto aleatorio por jugador) + Binomial Negativa para `receiving_tds`:
  **mejor R² de todos los intentos de este proyecto** para este target (+0.073, supera al hurdle
  y muy por encima del Tweedie), con MAE razonable.
- Totalmente interpretable sin SHAP: coeficientes fijos con intervalos de credibilidad
  (`depth_team` sale significativo, `wopr_last3_avg` no) y un efecto individual por jugador
  legible directamente (Ja'Marr Chase: ~+30% sobre un jugador promedio con sus mismas features).
- Se corrigió un problema real antes de ajustar: un `dropna()` ingenuo hubiera perdido 20.8% de
  las filas de entrenamiento, exactamente las primeras apariciones de temporada/carrera de cada
  jugador -- la población donde más importa el shrinkage jerárquico.
- Hallazgo honesto, contrario a la hipótesis inicial: el shrinkage no ayuda más a jugadores ya
  vistos en entrenamiento que a jugadores nuevos -- de hecho los nuevos salen con mejor MAE.
- Convergencia verificada sobre los 540 parámetros del modelo, no solo los efectos fijos --
  9 parámetros (todos efectos individuales de jugadores puntuales) quedan ligeramente sobre el
  umbral de r_hat, documentado sin ocultar.
- Modelo guardado con `joblib` (el formato nativo de `arviz` falló al recargar -- bug real
  documentado) y verificado con una recarga desde disco que reproduce las mismas métricas.
- **Siguiente paso natural, no resuelto aquí**: extender el mismo enfoque jerárquico a
  `receptions`/`receiving_yards` con la familia apropiada para cada uno (Binomial Negativa o
  Poisson para recepciones, Gaussiana o Gamma para yardas) -- decisión pendiente de Daniel antes
  de construirlo.